In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random_state = 42
np.random.seed(random_state)

In [ ]:
url = ''
df = pd.read_csv(url)

df.head()

In [ ]:
df.describe()

In [ ]:
df.boxplot()

In [ ]:
df.nunique()

In [ ]:
df.isna().sum()

In [ ]:
import seaborn as sns

sns.heatmap(df.corr(),annot=True)

In [ ]:
# encoder

# min max scalers

In [ ]:
target = ''
X = df.drop(target,axis=1)
y = df[target]

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.67,random_state=random_state)

In [ ]:
results = pd.DataFrame([],columns=['model','r2_score','RMSE'])

In [ ]:
# univariate
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,root_mean_squared_error

selected_feature = ''
X_train_uni = X_train[selected_feature]
X_test_uni = X_test[selected_feature]

lr = LinearRegression()
lr.fit(X_train_uni,y_train)
y_pred_lr_uni = lr.predict(X_test_uni)

# Coeff
coeff = lr.coef_[0]
intercept = lr.intercept_

results.loc[len(results)] = [
    f'Linear Regression Univariate on {selected_feature}',
    r2_score(y_test,y_pred_lr_uni),
    root_mean_squared_error(y_test,y_pred_lr_uni)
]


In [ ]:
# multivariate

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score,root_mean_squared_error

lr = LinearRegression()
lr.fit(X_train,y_train)
y_pred_multi = lr.predict(X_test)

# Coeff
coeff = lr.coef_
intercept = lr.intercept_


results.loc[len(results)] = [
    f'Linear Regression Multivariate',
    r2_score(y_test,y_pred_multi),
    root_mean_squared_error(y_test,y_pred_multi)
]


In [ ]:
# dt regressor

from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=random_state)
dt.fit(X_test,y_test)
max_depth = dt.tree_.max_dept

In [ ]:
from sklearn.model_selection import GridSearchCV

param_dt = {'max_depth':[*range(1,max_depth+1)]}

dt_gs = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=random_state),
    param_grid=param_dt,
    scoring='neg_mean_squared_error'
)
dt_gs.fit(X_train,y_train)
y_pred_dt = dt_gs.predict(X_test)

results = ....

In [ ]:
from sklearn.ensemble import RandomForestRegressor

param_rf = {'max_depth':[*range(4,max_depth+1)]}

rf_gs = GridSearchCV(
    estimator=RandomForestRegressor(random_state=random_state),
    param_grid=param_rf,
    scoring='neg_mean_squared_error'
)
rf_gs.fit(X_train,y_train)
y_pred_dt = rf_gs.predict(X_test)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

degrees = [*range(2,5)]
for degree in degrees:
    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.fit_transform(X_test)

    lr = LinearRegression()
    

In [ ]:
basket = df.groupby(['invoice_id','description'])['quantity'].sum()\
        .unstack()\
        .reset_index().fillna(0)\
        .set_index('invoice_id')

In [3]:
# 1. Create sample data
import pandas as pd
df_example = pd.DataFrame({
    'transaction_id': [1, 1, 1, 2, 2, 3, 3, 3],
    'element': ['Milk', 'Bread', 'Butter', 'Milk', 'Cheese', 'Bread', 'Butter', 'Eggs']
})

df_example.head()

,transaction_id,element
0,1,Milk
1,1,Bread
2,1,Butter
3,2,Milk
4,2,Cheese


In [12]:
# EXAMPLE: Improved basket creation with .size() and .astype(bool)

# 1. Create sample data
df_example = pd.DataFrame({
    'transaction_id': [1, 1, 1, 2, 2, 3, 3, 3],
    'element': ['Milk', 'Bread', 'Butter', 'Milk', 'Cheese', 'Bread', 'Butter', 'Eggs']
})

print("Original Data:")
print(df_example)
print("\n" + "="*50 + "\n")

# 2. Apply the improved basket creation
# NOTE: After .unstack(), transaction_id is ALREADY the index, no need to set_index again!
basket_example = (
    df_example.groupby(['transaction_id', 'element']).size()
    .unstack()
    .reset_index().fillna(0)
    .set_index('transaction_id')
    .astype(bool)
)

print("Basket (Binary - Presence/Absence):")
print(basket_example)
print("\nData Types:")
print(basket_example.dtypes)

Original Data:
   transaction_id element
0               1    Milk
1               1   Bread
2               1  Butter
3               2    Milk
4               2  Cheese
5               3   Bread
6               3  Butter
7               3    Eggs


Basket (Binary - Presence/Absence):
element         Bread  Butter  Cheese   Eggs   Milk
transaction_id                                     
1                True    True   False  False   True
2               False   False    True  False   True
3                True    True   False   True  False

Data Types:
element
Bread     bool
Butter    bool
Cheese    bool
Eggs      bool
Milk      bool
dtype: object


In [9]:
basket_example.head()

element,Bread,Butter,Cheese,Eggs,Milk
transaction_id,,,,,
1,True,True,False,False,True
2,False,False,True,False,True
3,True,True,False,True,False


In [5]:
# Step-by-step breakdown of the improved method

print("STEP 1: groupby(['transaction_id', 'element']).size()")
step1 = df_example.groupby(['transaction_id', 'element']).size()
print(step1)
print("\n" + "="*50 + "\n")

print("STEP 2: .unstack(fill_value=0)")
print("(Converts index to columns, fills missing with 0)")
step2 = step1.unstack(fill_value=0)
print(step2)
print("\n" + "="*50 + "\n")

print("STEP 3: .astype(bool)")
print("(Converts all non-zero values to True, zeros to False)")
step3 = step2.astype(bool)
print(step3)
print("\nData Types:")
print(step3.dtypes)

STEP 1: groupby(['transaction_id', 'element']).size()
transaction_id  element
1               Bread      1
                Butter     1
                Milk       1
2               Cheese     1
                Milk       1
3               Bread      1
                Butter     1
                Eggs       1
dtype: int64


STEP 2: .unstack(fill_value=0)
(Converts index to columns, fills missing with 0)
element         Bread  Butter  Cheese  Eggs  Milk
transaction_id                                   
1                   1       1       0     0     1
2                   0       0       1     0     1
3                   1       1       0     1     0


STEP 3: .astype(bool)
(Converts all non-zero values to True, zeros to False)
element         Bread  Butter  Cheese   Eggs   Milk
transaction_id                                     
1                True    True   False  False   True
2               False   False    True  False   True
3                True    True   False   True  False

Data